In [5]:
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [6]:
train_final = pd.read_csv(
    "../data/processed/zomato_train_engineered.csv"
)

test_final = pd.read_csv(
    "../data/processed/zomato_test_engineered.csv"
)

preprocessor = joblib.load(
    "../models/zomato_preprocessor.pkl"
)

print("Train shape:", train_final.shape)
print("Test shape :", test_final.shape)
print("Preprocessor loaded successfully.")

Train shape: (33332, 27)
Test shape : (8333, 27)
Preprocessor loaded successfully.


In [7]:
feature_columns = [
    "online_order",
    "book_table",
    "approx_costfor_two_people",
    "log_cost",
    "cost_band",
    "location",
    "primary_cuisine",
    "cuisine_count",
    "primary_rest_type",
    "historical_restaurant_count",
    "location_median_cost",
    "location_online_order_rate",
    "location_book_table_rate",
    "location_cuisine_diversity",
    "location_business_type_diversity"
]

X_train = train_final[feature_columns].copy()
X_test = test_final[feature_columns].copy()

y_train = train_final["performance_class"].copy()
y_test = test_final["performance_class"].copy()

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (33332, 15)
X_test : (8333, 15)
y_train: (33332,)
y_test : (8333,)


In [8]:
X_train_encoded = preprocessor.transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Encoded X_train:", X_train_encoded.shape)
print("Encoded X_test :", X_test_encoded.shape)

Encoded X_train: (33332, 219)
Encoded X_test : (8333, 219)


In [9]:
print("NaNs in X_train:",
      np.isnan(X_train_encoded).sum())

print("NaNs in X_test:",
      np.isnan(X_test_encoded).sum())

print("Infinite values in X_train:",
      np.isinf(X_train_encoded).sum())

print("Infinite values in X_test:",
      np.isinf(X_test_encoded).sum())

NaNs in X_train: 0
NaNs in X_test: 0
Infinite values in X_train: 0
Infinite values in X_test: 0


In [10]:
print("\nTraining target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())


Training target:
performance_class
Medium    11331
Low       11002
High      10999
Name: count, dtype: int64

Test target:
performance_class
High      2952
Medium    2795
Low       2586
Name: count, dtype: int64


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_encoded)
X_test_scaled = scaler.transform(X_test_encoded)

print("Scaled train:", X_train_scaled.shape)
print("Scaled test :", X_test_scaled.shape)

Scaled train: (33332, 219)
Scaled test : (8333, 219)


In [13]:
logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model.fit(
    X_train_scaled,
    y_train
)

logistic_pred = logistic_model.predict(
    X_test_scaled
)

In [14]:
print(
    "Logistic Regression Accuracy:",
    accuracy_score(y_test, logistic_pred)
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        logistic_pred
    )
)

Logistic Regression Accuracy: 0.5814232569302772

Classification Report:
              precision    recall  f1-score   support

        High       0.73      0.66      0.69      2952
         Low       0.55      0.65      0.59      2586
      Medium       0.47      0.44      0.45      2795

    accuracy                           0.58      8333
   macro avg       0.58      0.58      0.58      8333
weighted avg       0.59      0.58      0.58      8333



Random Forest

Random Forest is evaluated as a nonlinear tree-based model.

Unlike Logistic Regression, Random Forest can capture nonlinear relationships and interactions between business and locality features.

The model is trained using the encoded features without standardization.

In [15]:
random_forest = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

random_forest.fit(
    X_train_encoded,
    y_train
)

rf_pred = random_forest.predict(
    X_test_encoded
)

In [16]:
print(
    "Random Forest Accuracy:",
    accuracy_score(y_test, rf_pred)
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        rf_pred
    )
)

Random Forest Accuracy: 0.8702748109924398

Classification Report:
              precision    recall  f1-score   support

        High       0.95      0.87      0.91      2952
         Low       0.84      0.90      0.87      2586
      Medium       0.83      0.84      0.83      2795

    accuracy                           0.87      8333
   macro avg       0.87      0.87      0.87      8333
weighted avg       0.87      0.87      0.87      8333



In [17]:
rf_f1 = f1_score(
    y_test,
    rf_pred,
    average="macro"
)

print("Random Forest Macro F1:", rf_f1)

Random Forest Macro F1: 0.870234562199364


Random Forest Feature Importance

Feature importance is examined to understand which engineered variables contribute most strongly to the Random Forest predictions.

This provides model interpretability and helps identify whether the model is relying on meaningful business and locality characteristics.

The importance analysis will also be used to detect potentially problematic or unexpectedly dominant features.

In [18]:
encoded_feature_names = preprocessor.get_feature_names_out()

print("Number of feature names:", len(encoded_feature_names))

Number of feature names: 219


In [19]:
feature_importance = pd.DataFrame({
    "feature": encoded_feature_names,
    "importance": random_forest.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

display(
    feature_importance.head(30)
)

,feature,importance
212,numerical__cuisine_count,0.114712
210,numerical__approx_costfor_two_people,0.094249
211,numerical__log_cost,0.091128
216,numerical__location_book_table_rate,0.039540
3,categorical__book_table_1,0.033070
213,numerical__historical_restaurant_count,0.032977
215,numerical__location_online_order_rate,0.032398
217,numerical__location_cuisine_diversity,0.029722
2,categorical__book_table_0,0.027755
214,numerical__location_median_cost,0.026169


 Permutation Feature Importance

Random Forest impurity-based importance provides an initial estimate of feature contribution.

Permutation importance is used as a second validation method.

For each feature, its values are randomly shuffled in the test set while the remaining features remain unchanged.

A large decrease in model performance indicates that the model relies strongly on that feature.

Permutation importance is calculated on the held-out test set and is used only for model interpretation.

In [20]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    random_forest,
    X_test_encoded,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="f1_macro",
    n_jobs=-1
)

In [21]:
perm_importance = pd.DataFrame({
    "feature": encoded_feature_names,
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(
    "importance_mean",
    ascending=False
)

display(
    perm_importance.head(30)
)

,feature,importance_mean,importance_std
212,numerical__cuisine_count,0.078676,0.001613
211,numerical__log_cost,0.028482,0.000865
210,numerical__approx_costfor_two_people,0.026320,0.000665
162,categorical__primary_cuisine_North Indian,0.022495,0.001458
1,categorical__online_order_1,0.022032,0.001738
207,categorical__primary_rest_type_Quick Bites,0.011203,0.000996
0,categorical__online_order_0,0.010891,0.001778
193,categorical__primary_rest_type_Casual Dining,0.010355,0.000999
176,categorical__primary_cuisine_South Indian,0.010258,0.000354
123,categorical__primary_cuisine_Chinese,0.008166,0.000579
